# 대한민국 출생 통계 분석

KOSIS 엑셀 원본을 pandas로 읽어 데이터 품질을 확인하고, 분석하기 쉬운 연도별 표로 정리합니다. 출생아 수·합계출산율·자연증가의 장기 추세, 연대별 평균, 최근 연간 변화와 지표 간 상관관계를 살펴보고 출생아 수 라인 그래프를 만듭니다.

- 대상 기간: 1970~2025년
- 출처: 원본 엑셀의 KOSIS 통계표 정보
- 산출물: `birth_statistics_cleaned.csv`, `births_by_year.png`

> 자연증가의 음수 값은 오류가 아니라 사망자 수가 출생아 수보다 많은 해를 뜻하므로 그대로 둡니다. 결측값도 임의로 채우지 않습니다.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib import font_manager
from matplotlib.ticker import FuncFormatter

SOURCE_FILE = next(Path.cwd().glob('*20261030140727.xlsx'))
OUTPUT_DIR = Path.cwd()
print(f'원본 파일: {SOURCE_FILE.name}')
print(f'pandas {pd.__version__}')

## 1. 원본 구조 확인

첫 번째 시트는 첫 행에 연도, 첫 열에 지표명이 있는 넓은(wide) 형태입니다. 메타데이터 시트는 분석 데이터에서 제외합니다.

In [ ]:
raw = pd.read_excel(SOURCE_FILE, sheet_name=0, header=None)
print('원본 크기:', raw.shape)
display(raw.iloc[:7, :6])

## 2. 데이터 클렌징 및 품질 점검

연도와 측정값을 숫자로 변환하고, 지표를 행에서 열로 옮겨 한 행이 한 해를 나타내도록 정리합니다. 변환 과정에서 생긴 결측은 원본 결측과 구분해 확인하며, 중복 연도는 자동 제거하지 않고 오류로 알립니다.

In [ ]:
years = pd.to_numeric(raw.iloc[0, 1:], errors='raise').astype('int64')
metric_names = raw.iloc[1:, 0].astype(str).str.strip().tolist()
source_values = raw.iloc[1:, 1:].T.copy()
source_values.columns = metric_names
source_values.index = pd.Index(years, name='연도')

# 엑셀 셀의 수치와 원래 비어 있던 셀을 구분해, 잘못된 텍스트 변환을 드러냅니다.
cleaned = source_values.apply(pd.to_numeric, errors='coerce')
conversion_failures = source_values.notna() & cleaned.isna()
failure_count = int(conversion_failures.to_numpy().sum())
if failure_count:
    print('숫자로 변환되지 않은 원본 셀:', failure_count)
    display(source_values.where(conversion_failures).stack())

if cleaned.index.has_duplicates:
    raise ValueError('중복 연도가 있어 분석을 중단합니다.')
if cleaned.columns.has_duplicates:
    raise ValueError('중복 지표명이 있어 분석을 중단합니다.')

cleaned = cleaned.sort_index()
print('정제 데이터 크기:', cleaned.shape)
print('연도 범위:', int(cleaned.index.min()), '~', int(cleaned.index.max()))
print('결측치 수(지표별):')
display(cleaned.isna().sum().rename('결측치'))
print('중복 연도 수:', int(cleaned.index.duplicated().sum()))
display(cleaned.head())

cleaned.to_csv(OUTPUT_DIR / 'birth_statistics_cleaned.csv', encoding='utf-8-sig')

## 3. 다각도 분석

기술통계와 지표별 최솟값·최댓값, 첫해와 마지막 해의 변화, 연대별 평균, 최근 연간 증감률, 지표 간 상관계수를 계산합니다. 연대별 평균에서 2020년대는 2020~2025년만 포함하는 부분 기간입니다.

In [ ]:
birth_col, natural_col, crude_col, natural_rate_col, tfr_col, sex_ratio_col = cleaned.columns

print('기술통계')
display(cleaned.describe().T.round(3))

first_year, last_year = int(cleaned.index.min()), int(cleaned.index.max())
trend_rows = []
for col in cleaned.columns:
    start, end = cleaned[col].iloc[0], cleaned[col].iloc[-1]
    pct = (end / start - 1) * 100 if start != 0 and start * end > 0 else np.nan
    trend_rows.append({
        '지표': col, '시작 연도': first_year, '시작 값': start,
        '마지막 연도': last_year, '마지막 값': end,
        '절대 변화': end - start, '변화율(%)': pct,
        '최솟값 연도': int(cleaned[col].idxmin()), '최솟값': cleaned[col].min(),
        '최댓값 연도': int(cleaned[col].idxmax()), '최댓값': cleaned[col].max(),
    })
trend = pd.DataFrame(trend_rows).set_index('지표')
print(f'{first_year}년 대비 {last_year}년')
display(trend.round(3))

decade = cleaned.assign(연대=(cleaned.index // 10) * 10)
decade_means = decade.groupby('연대')[[birth_col, tfr_col, natural_col]].mean()
print('연대별 평균 (2020년대는 2020~2025년)')
display(decade_means.round(2))

yoy = cleaned[[birth_col, tfr_col, natural_col]].pct_change() * 100
print('최근 10년의 출생아 수 연간 증감률(%)')
display(yoy[birth_col].tail(10).rename('증감률(%)').round(2))
print('마지막 2개 연도의 주요 지표')
display(cleaned[[birth_col, tfr_col, natural_col]].tail(2))

print('상관계수 (시간 추세를 반영할 수 있으며 인과관계를 뜻하지 않음)')
display(cleaned.corr().round(3))
print('자연증가가 음수인 연도:', cleaned.index[cleaned[natural_col] < 0].tolist())

## 4. 연도별 출생아 수 라인 그래프

전체 연도의 흐름을 보이면서 최고·최저 기록을 표시합니다. 그래프는 별도 PNG로 저장합니다.

In [ ]:
births = cleaned[birth_col]
peak_year, low_year = int(births.idxmax()), int(births.idxmin())
korean_font = Path(r'C:\Windows\Fonts\malgun.ttf')
use_korean = korean_font.exists()
if use_korean:
    font_manager.fontManager.addfont(korean_font)
    plt.rcParams['font.family'] = font_manager.FontProperties(fname=korean_font).get_name()
    plt.rcParams['axes.unicode_minus'] = False
fig, ax = plt.subplots(figsize=(12, 6))
ax.plot(births.index, births.values, color='#1769aa', linewidth=2, marker='o', markersize=3)
ax.scatter([peak_year, low_year], [births.loc[peak_year], births.loc[low_year]],
           color=['#d1495b', '#f28e2b'], zorder=3)
ax.annotate(f'{"최고" if use_korean else "Peak"}: {peak_year} ({births.loc[peak_year]:,.0f})',
            (peak_year, births.loc[peak_year]), xytext=(8, -24),
            textcoords='offset points', color='#9b2636')
ax.annotate(f'{"최저" if use_korean else "Low"}: {low_year} ({births.loc[low_year]:,.0f})',
            (low_year, births.loc[low_year]), xytext=(8, 10),
            textcoords='offset points', color='#a85a00')
ax.set_title(f'대한민국 연도별 출생아 수 ({first_year}-{last_year})' if use_korean else
             f'Live births in South Korea ({first_year}-{last_year})')
ax.set_xlabel('연도' if use_korean else 'Year')
ax.set_ylabel('출생아 수(명)' if use_korean else 'Live births (people)')
ax.yaxis.set_major_formatter(FuncFormatter(lambda value, _: f'{value:,.0f}'))
ax.set_xlim(first_year, last_year)
ax.grid(True, alpha=0.25)
fig.tight_layout()
chart_path = OUTPUT_DIR / 'births_by_year.png'
fig.savefig(chart_path, dpi=180, bbox_inches='tight')
plt.show()
plt.close(fig)
print('그래프 저장:', chart_path)

## 해석 시 유의점

- 출생아 수와 합계출산율은 서로 다른 지표입니다. 출생아 수는 연간 출생 규모, 합계출산율은 여성 1명이 평생 낳을 것으로 예상되는 평균 자녀 수를 나타냅니다.
- 자연증가에는 출생뿐 아니라 사망도 반영됩니다. 값이 음수인 해는 자연감소를 뜻합니다.
- 상관계수는 동반 변화의 정도만 보여 주며, 인과관계를 증명하지 않습니다.